<!-- NOTEBOOK_METADATA source: "⚠️ Jupyter Notebook" title: "Bifrost AI Gateway Integration" sidebarTitle: "Bifrost" logo: "/images/integrations/bifrost_icon.png" description: "Learn how to send OpenTelemetry traces from the Bifrost AI gateway to Langfuse while using its OpenAI-compatible API." category: "Integrations" -->

# Integrate Langfuse with Bifrost

This notebook shows how to connect Langfuse to the Bifrost AI gateway for tracing and observability of OpenAI-compatible LLM requests.

> **What is Bifrost?** [Bifrost](https://github.com/maximhq/bifrost) is an open-source, self-hosted Go AI gateway that provides an OpenAI-compatible API with multi-provider routing, adaptive load balancing, guardrails, and virtual keys.

> **What is Langfuse?** [Langfuse](https://langfuse.com) is an open-source LLM engineering platform that helps teams trace, debug, and evaluate their LLM applications.

<!-- STEPS_START -->
## Step 1: Install Dependencies

In [ ]:
%pip install langfuse openai -U

## Step 2: Set Up Environment Variables

Get your Langfuse keys from the project settings in [Langfuse Cloud](https://langfuse.com/cloud) or set up [self-hosting](https://langfuse.com/self-hosting).

In [ ]:
import os

# Get keys for your project from the project settings page: https://langfuse.com/cloud
os.environ.setdefault("LANGFUSE_PUBLIC_KEY", "pk-lf-...");
os.environ.setdefault("LANGFUSE_SECRET_KEY", "sk-lf-...");
os.environ.setdefault("LANGFUSE_BASE_URL", "https://cloud.langfuse.com"); # 🇪🇺 EU region (API host)
# Other Langfuse data regions include 🇺🇸 US: https://us.cloud.langfuse.com, 🇯🇵 Japan: https://jp.cloud.langfuse.com and ⚕️ HIPAA: https://hipaa.cloud.langfuse.com

os.environ.setdefault("BIFROST_BASE_URL", "http://localhost:8080/v1");  # OpenAI-compatible Bifrost endpoint.
os.environ.setdefault("BIFROST_API_KEY", "your-bifrost-virtual-key");  # Create a Bifrost virtual key and use its value here.

With the environment variables set, initialize the Langfuse client. `get_client()` picks up the env vars above and returns a client bound to your project.


In [ ]:
from langfuse import get_client

langfuse = get_client()

# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

## Step 3: Configure Bifrost OpenTelemetry Export

Before starting the gateway, follow the [Bifrost gateway quickstart](https://docs.getbifrost.ai/quickstart/gateway/setting-up) and configure at least one provider key in config.json using the [provider configuration guide](https://docs.getbifrost.ai/deployment-guides/config-json/providers). Create a virtual key and use its value for BIFROST_API_KEY with the [virtual keys guide](https://docs.getbifrost.ai/features/governance/virtual-keys). Keep the gateway on http://localhost:8080.

The code below writes only the OpenTelemetry plugin block to bifrost-config.json. Merge the plugins block into the config.json used by your Bifrost deployment and keep your existing provider and governance settings. See Bifrost's [plugin configuration](https://docs.getbifrost.ai/deployment-guides/config-json/plugins) and [environment variable references](https://docs.getbifrost.ai/deployment-guides/config-json#environment-variable-references).

In [ ]:
import json
import os
from pathlib import Path

bifrost_config = {
    "plugins": [
        {
            "enabled": True,
            "name": "otel",
            "config": {
                "service_name": "bifrost",
                "collector_url": f"{os.environ['LANGFUSE_BASE_URL'].rstrip('/')}/api/public/otel/v1/traces",
                "trace_type": "genai_extension",
                "protocol": "http",
                "headers": {
                    "Authorization": "env.LANGFUSE_OTEL_AUTH",
                    "x-langfuse-ingestion-version": "4",
                },
            },
        }
    ]
}

Path("bifrost-config.json").write_text(
    json.dumps(bifrost_config, indent=2), encoding="utf-8"
)
print("Wrote bifrost-config.json. Merge its plugins block into the config.json used by Bifrost.")
print("Bifrost must receive LANGFUSE_OTEL_AUTH in its own launch environment; do not put the encoded secret in this file.")

## Step 4: Start Bifrost with Langfuse Authentication

The notebook kernel and an independently started Bifrost process do not share later environment changes. Bifrost must inherit LANGFUSE_OTEL_AUTH from the same shell or container where it starts.

After merging the plugins block, set your Langfuse keys and start Bifrost from that environment. On macOS or Linux:

~~~bash
export LANGFUSE_PUBLIC_KEY="pk-lf-..."
export LANGFUSE_SECRET_KEY="sk-lf-..."
export OPENAI_API_KEY="sk-..."
export BIFROST_ENCRYPTION_KEY="replace-with-a-32-byte-key"
export LANGFUSE_OTEL_AUTH="Basic $(printf '%s' "$LANGFUSE_PUBLIC_KEY:$LANGFUSE_SECRET_KEY" | base64)"
npx -y @maximhq/bifrost -app-dir ./data
~~~

On PowerShell:

~~~powershell
$pair = "$($env:LANGFUSE_PUBLIC_KEY):$($env:LANGFUSE_SECRET_KEY)"
$bytes = [Text.Encoding]::UTF8.GetBytes($pair)
$env:LANGFUSE_OTEL_AUTH = "Basic " + [Convert]::ToBase64String($bytes)
npx -y @maximhq/bifrost -app-dir ./data
~~~

For Docker, pass LANGFUSE_OTEL_AUTH, OPENAI_API_KEY, and BIFROST_ENCRYPTION_KEY with -e when starting the container. Keep Bifrost running at http://localhost:8080 before executing the next step.

## Step 5: Send a Request Through Bifrost

With Bifrost running and its virtual key in BIFROST_API_KEY, send an OpenAI-compatible request through the gateway.

In [ ]:
import os

from openai import OpenAI

# Bifrost routes this OpenAI-compatible request to the configured provider.
client = OpenAI(
    base_url=os.environ["BIFROST_BASE_URL"],
    api_key=os.environ["BIFROST_API_KEY"],
)

response = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Explain why tracing routed LLM requests is useful."}
    ],
)

print(response.choices[0].message.content)

## Step 6: View Traces in Langfuse

After running the example, open [Langfuse Cloud](https://langfuse.com/cloud) to see the full trace including prompts, completions, tool calls, token usage, and latency.

![Example Bifrost trace in Langfuse](https://langfuse.com/images/cookbook/integration-bifrost/bifrost-example-trace.png)

A public example trace link is omitted because traces are project-specific. Run the example in your Langfuse project to open the trace.

<!-- STEPS_END -->

<!-- MARKDOWN_COMPONENT name: "LearnMore" path: "@/components-mdx/integration-learn-more.mdx" -->